# 🦾 Use Llava-NeXT-Video with JSON adapter

Training Toolkit is a framework focused on training LLM adapters. However, it would be cruel of us to leave you without any inference code at all.

Assuming you already got you trained adapter for Llava-NeXT-Video, here's a simple walkthough of using it for inference.

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import sys


sys.path.append(Path("..").resolve().as_posix())
_ = load_dotenv()

## 1. Load the model

- HF 🤗 PEFT is going to identify the base model, load it, load the adapter and attach the two together.
- We need a special utility to load the video. Let's use the same one that we used during training

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoProcessor
from training_toolkit.common.video_readers import get_video_reader


CHECKPOINT_PATH = "llava_next_video/adapter/checkpoint"

model = AutoPeftModelForCausalLM.from_pretrained(CHECKPOINT_PATH)
processor = AutoProcessor.from_pretrained(CHECKPOINT_PATH)
video_reader = get_video_reader()

## 2. Load and preprocess the sample

In [ ]:
import torch

# 1. Load the video
video = torch.tensor(
    video_reader(
        "path/to/video.mp4",
        num_frames=8,
    )
)

# 2. Prepare the prompt
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "extract JSON."},
            {"type": "video"},
        ],
    },
]

prompt = processor.apply_chat_template(conversation, add_generation_prompt=False)

# 3. Process the video and tokenize the prompt
inputs = processor(
    text=prompt,
    videos=video,
    truncation=True,
    max_length=1024,
    return_tensors="pt",
)

## 3. Generate the JSON

In [ ]:
# Generate text

generated_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=True)

generated_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)[0]